In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()


In [0]:
#skewed dataset 
df_skewed = spark.createDataFrame([(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'e'), (3, 'f')], ['id', 'value'])
#, 'c'), (4, 'd'), (5, 'e'), (6, 'f')], ['id', 'value'])
df_skewed.show()
#lookup dataset
df_lookup = spark.createDataFrame([(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')], ['id', 'lookup_value'])
df_lookup.show()


In [0]:
#salting
salt_count = 4
df_skewed_slated = df_skewed.withColumn("salt",(F.rand()*salt_count).cast("int"))
df_skewed_salted = df_skewed_slated.withColumn("id_salted",F.concat_ws("_",F.col("id"),F.col("salt")))
df_skewed_salted.show()

In [0]:
#explode same for lookup table
salt_array = F.sequence(F.lit(0),F.lit(salt_count-1))
df_lookup_salted = df_lookup.withColumn("salt",F.explode(salt_array))
df_lookup_salted = df_lookup_salted.withColumn("id_salted",F.concat_ws("_",F.col("id"),F.col("salt")))
final_join = df_skewed_salted.join(df_lookup_salted,on=["id_salted","id"]).drop("id_salted","salt")
final_join.show()

* Find duplicate elements in list

In [0]:
numbers = [1,2,3,2,4,5,1,6]
dups=[]
unique=[]
for num in numbers:
    if numbers.count(num)>1:
        dups.append(num)
    else:
        unique.append(num)
print(dups)
print(unique)

In [0]:
numbers = [1,1,1,2,3,4,5]
dict ={}
for num in numbers:
    if numbers.count(num)>1:
        print(dict)
        dict[num]=numbers.count(num)
    else:
        dict[num]=1
print(dict)

# List vs Tuple vs Set vs Dictionary

| Data Structure | Ordered? | Mutable? | Duplicates? | Syntax | Use when... |
|---|---|---|---|---|---|
| **List** | ✅ Yes | ✅ Yes (add/remove/change) | ✅ Allowed | `[1, 2, 3]` | You need an ordered collection that changes over time |
| **Tuple** | ✅ Yes | ❌ No (immutable) | ✅ Allowed | `(1, 2, 3)` | You want fixed, read-only data (safer + slightly faster) |
| **Set** | ❌ No (unordered) | ✅ Yes (add/remove) | ❌ Not allowed | `{1, 2, 3}` | You need unique values / fast membership tests / set math |
| **Dictionary** | ✅ Insertion-ordered | ✅ Yes (add/update/remove) | ❌ Keys must be unique | `{"a": 1}` | You need key → value lookups (fast access by key) |

## Quick examples

```python
# List — ordered, mutable, allows duplicates
numbers = [1, 2, 2, 3]
numbers.append(4)          # [1, 2, 2, 3, 4]

# Tuple — ordered, immutable, allows duplicates
point = (10, 20, 20)
# point[0] = 99           # ❌ TypeError: tuples can't be changed

# Set — unordered, mutable, NO duplicates
unique = {1, 2, 2, 3}     # {1, 2, 3} — duplicate 2 dropped automatically
unique.add(4)

# Dictionary — key/value pairs, keys must be unique
ages = {"a": 1, "b": 2}
ages["c"] = 3              # add a new key
print(ages["b"])           # 2 — fast lookup by key
```

## Key takeaways

* **List vs tuple**: same behavior except tuples are **immutable** — use tuples for fixed records (like `point`), lists for data that changes.
* **Set** automatically removes duplicates and supports operations like `union`, `intersection`, `difference`.
* **Dictionary** maps **keys to values**; lookup by key is very fast (O(1)), unlike searching a list (O(n)).
* Only tuples can be used as **dictionary keys** (lists/sets can't, because they're unhashable/mutable).

In [0]:
text ="python spark python sql python aws"
d={}
for word in text.split(" "):
    if word in d:
        d[word]+=1
    else:
        d[word]=1
print(d)

In [0]:
d1={}
for word in text.split(" "):
    d1[word] = d1.get(word,0)+1
print(d1)

In [0]:
from collections import Counter
d2 = Counter(text.split(" "))
print(d2)

In [0]:
import copy

# Shallow copy vs Deep copy

# Shallow copy -> copies the outer object, but INNER (nested) objects are still SHARED (same references)
# Deep copy    -> copies the outer object AND recursively copies all nested objects (fully independent)

# ---------- Shallow copy ----------
original = [[1, 2, 3], [4, 5, 6]]
shallow = copy.copy(original)          # also works: original[:] or list(original) or original.copy()

shallow[0][0] = 99                     # change a NESTED element in the copy
print("original after shallow copy change:", original)  # [[99, 2, 3], [4, 5, 6]]  <-- original changed too!
print("shallow:", shallow)

# Deep copy
original2 = [[1, 2, 3], [4, 5, 6]]
deep = copy.deepcopy(original2)

deep[0][0] = 99                        # change a NESTED element in the copy
print("original2 after deep copy change:", original2)   # [[1, 2, 3], [4, 5, 6]]  <-- original unchanged
print("deep:", deep)

# ---------- Verify identity ----------
shallow2 = copy.copy(original2)
deep2 = copy.deepcopy(original2)

print(shallow2[0] is original2[0])   # True  -> inner list is the SAME object (shallow)
print(deep2[0] is original2[0])      # False -> inner list is a NEW object (deep)

# Key takeaways:
# * copy.copy(x)   : new outer object, nested objects shared -> mutating nested data affects both
# * copy.deepcopy(x): fully independent clone -> safe to modify nested structures
# * For flat lists of immutables (e.g. [1,2,3]), shallow and deep copy behave the same
